In [64]:
import pandas as pd
import numpy as np
import json
import re

# Load dataset using pandas
with open("jobs_v2.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(df.shape)

(585, 15)


In [65]:
df_before = df.copy()

In [66]:
# ---------- Column names ----------
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("&", "and", regex=False)
)

In [67]:
# ---------- Clean text ----------
exclude_cols = ["skills_and_tools", "experience_needed"]
text_cols = df.select_dtypes(include=["object", "string"]).columns
strip_cols = [c for c in text_cols if c not in exclude_cols]

for col in strip_cols:
    df[col] = df[col].str.strip()


In [68]:
# ---------- Missing values ----------
for col in strip_cols:
    df[col] = df[col].replace(["N/A", ""], np.nan)

for col in strip_cols:
    if col != "salary":
        df[col] = df[col].fillna("Unknown")

In [69]:
# ---------- Remove duplicates ----------
df = df.drop_duplicates(subset="url").reset_index(drop=True)

In [70]:
#  Arabic location translation ----------
arabic_location_map = {
    "هليوبوليس, القاهرة, مصر": "Heliopolis, Cairo, Egypt",
    "القاهرة الجديدة, القاهرة, مصر": "New Cairo, Cairo, Egypt",
}
df["location"] = df["location"].replace(arabic_location_map)

In [71]:
# ---------- Company size ----------
arabic_company_size = {
    "١١ - ٥٠ موظف": "11-50 employees",
    "٥٠١ - ١٠٠٠ موظف": "501-1000 employees",
}
df["company_size"] = df["company_size"].replace(arabic_company_size)

def parse_company_size(s):
    if pd.isna(s) or s == "Unknown":
        return np.nan, np.nan, np.nan

    s = str(s)

    more = re.search(r"more than\s+(\d+)", s, re.IGNORECASE)
    if more:
        v = int(more.group(1))
        return v, v, float(v)

    nums = re.findall(r"\d+", s)
    if len(nums) == 1:
        v = int(nums[0])
        return v, v, float(v)

    if len(nums) >= 2:
        lo, hi = int(nums[0]), int(nums[1])
        return lo, hi, (lo + hi) / 2

    return np.nan, np.nan, np.nan

# Keep the original company_size column, add min/max/avg alongside it
df[["company_size_min", "company_size_max", "company_size_avg"]] = (
    df["company_size"].apply(parse_company_size).apply(pd.Series)
)

median_size = df["company_size_avg"].median()
df["company_size_avg"] = df["company_size_avg"].fillna(median_size)
df["company_size_min"] = df["company_size_min"].fillna(median_size)
df["company_size_max"] = df["company_size_max"].fillna(median_size)

# Cast min/max/avg to integer (employees are whole numbers)
df["company_size_avg"] = df["company_size_avg"].round().astype(int)
df["company_size_min"] = df["company_size_min"].round().astype(int)
df["company_size_max"] = df["company_size_max"].round().astype(int)


In [72]:
# ---------- Experience ----------
def extract_experience(x):
    if isinstance(x, list) and len(x) >= 2:
        min_exp = x[0] if isinstance(x[0], (int, float)) else np.nan
        max_exp = x[1] if isinstance(x[1], (int, float)) else np.nan
        return min_exp, max_exp
    return np.nan, np.nan


df[["min_experience_years", "max_experience_years"]] = (
    df["experience_needed"].apply(extract_experience).apply(pd.Series)
)

df.drop(columns="experience_needed", inplace=True)

median_max_exp = df["max_experience_years"].median()
median_min_exp = df["min_experience_years"].median()
df["max_experience_years"] = df["max_experience_years"].fillna(median_max_exp)
df["min_experience_years"] = df["min_experience_years"].fillna(median_min_exp)

In [73]:
from IPython.display import display

sample_idx = df_before.sample(5, random_state=43).index

print("Before:")
display(df_before.loc[sample_idx])

print("After:")
display(df.loc[sample_idx])

Before:


,Job Title,Company Name,Company Size,Location,Work Type,Work Setting,Experience Needed,Career Level,Education Level,Salary,Skills & Tools,Job Description,Job Requirements,Posted At,url
550,Laravel Developer,Al Ghad TV,101-500 employees,"6th of October, Giza, Egypt",Full Time,On-site,"[3, 7]",Experienced (Non-Manager),Bachelor's Degree,N/A,"[PHP+, Laravel Framework, Laravel REST APIs, L...",Role Overview: The successful candidate will ...,Preferred Qualifications: Experience with h...,04/07/2026 00:48:14,https://wuzzuf.net/jobs/p/lokww6j945fm-laravel...
170,Software Developer,Renewable Energy Pros,11-50 employees,"New Cairo, Cairo, Egypt",Full Time,On-site,"[2, 5]",Experienced (Non-Manager),Bachelor's Degree,N/A,"[Python, Software Development, Django Framewor...",Job Description Your Impact: As a Softwar...,Basic Qualifications Bachelor's degree in C...,03/04/2026 20:43:27,https://wuzzuf.net/jobs/p/kigtdoolgavl-softwar...
446,Senior PHP Developer & Technical Lead,BimmerTech,N/A,"Dubai, United Arab Emirates",Full Time,N/A,N/A,Experienced (Non-Manager),Not Specified,N/A,"[Information Technology (IT), Computer Science...",About The RoleWe are seeking a highly skilled ...,,03/13/2026 03:16:38,https://wuzzuf.net/jobs/p/g/ajjdkzlo2r3d-senio...
551,Senior Frontend Developer,FirstTech,101-500 employees,"Nasr City, Cairo, Egypt",Full Time,Hybrid,"[8, 15]",Experienced (Non-Manager),Bachelor's Degree,N/A,"[TypeScript, React, React Native, JavaScript, ...","Responsible for building performant, Single Pa...",Bachelor's degree in Engineering or Technology...,04/07/2026 09:23:01,https://wuzzuf.net/jobs/p/dgse98psxe5o-senior-...
523,IT Engineer,N/A,N/A,"Giza, Giza, Egypt",Full Time,Hybrid,"[1, 2]",Entry Level (Junior Level / Fresh Grad),Bachelor's Degree,N/A,"[Information Technology (IT), Engineering Prin...",Unleash Your Engineering Prowess as an Integra...,Exciting Qualifications for a Transformative R...,03/30/2026 12:41:48,https://wuzzuf.net/jobs/p/iys5efvox4as-it-engi...


After:


,job_title,company_name,company_size,location,work_type,work_setting,career_level,education_level,salary,skills_and_tools,job_description,job_requirements,posted_at,url,company_size_min,company_size_max,company_size_avg,min_experience_years,max_experience_years
550,Laravel Developer,Al Ghad TV,101-500 employees,"6th of October, Giza, Egypt",Full Time,On-site,Experienced (Non-Manager),Bachelor's Degree,NaN,"[PHP+, Laravel Framework, Laravel REST APIs, L...",Role Overview: The successful candidate will ...,Preferred Qualifications: Experience with h...,04/07/2026 00:48:14,https://wuzzuf.net/jobs/p/lokww6j945fm-laravel...,101,500,300,3.0,7.0
170,Software Developer,Renewable Energy Pros,11-50 employees,"New Cairo, Cairo, Egypt",Full Time,On-site,Experienced (Non-Manager),Bachelor's Degree,NaN,"[Python, Software Development, Django Framewor...",Job Description Your Impact: As a Softwar...,Basic Qualifications Bachelor's degree in C...,03/04/2026 20:43:27,https://wuzzuf.net/jobs/p/kigtdoolgavl-softwar...,11,50,30,2.0,5.0
446,Senior PHP Developer & Technical Lead,BimmerTech,Unknown,"Dubai, United Arab Emirates",Full Time,Unknown,Experienced (Non-Manager),Not Specified,NaN,"[Information Technology (IT), Computer Science...",About The RoleWe are seeking a highly skilled ...,Unknown,03/13/2026 03:16:38,https://wuzzuf.net/jobs/p/g/ajjdkzlo2r3d-senio...,76,76,76,3.0,6.0
551,Senior Frontend Developer,FirstTech,101-500 employees,"Nasr City, Cairo, Egypt",Full Time,Hybrid,Experienced (Non-Manager),Bachelor's Degree,NaN,"[TypeScript, React, React Native, JavaScript, ...","Responsible for building performant, Single Pa...",Bachelor's degree in Engineering or Technology...,04/07/2026 09:23:01,https://wuzzuf.net/jobs/p/dgse98psxe5o-senior-...,101,500,300,8.0,15.0
523,IT Engineer,Unknown,Unknown,"Giza, Giza, Egypt",Full Time,Hybrid,Entry Level (Junior Level / Fresh Grad),Bachelor's Degree,NaN,"[Information Technology (IT), Engineering Prin...",Unleash Your Engineering Prowess as an Integra...,Exciting Qualifications for a Transformative R...,03/30/2026 12:41:48,https://wuzzuf.net/jobs/p/iys5efvox4as-it-engi...,76,76,76,1.0,2.0


In [74]:
# ---------- Salary ----------
def clean_salary(s):
    if pd.isna(s):
        return np.nan, np.nan, np.nan, np.nan

    s = re.sub(r",?\s*Bonus.*", "", str(s), flags=re.IGNORECASE)

    m = re.search(r"(\d[\d,]*)\s+to\s+(\d[\d,]*)\s+(\w+)\s+Per\s+(\w+)", s)
    if m:
        return (
            float(m.group(1).replace(",", "")),
            float(m.group(2).replace(",", "")),
            m.group(3).upper(),
            m.group(4).capitalize(),
        )

    return np.nan, np.nan, np.nan, np.nan


df[["salary_min", "salary_max", "salary_currency", "salary_period"]] = (
    df["salary"].apply(clean_salary).apply(pd.Series)
)

df.drop(columns="salary", inplace=True)

print(f"Salaries disclosed: {df['salary_min'].notna().sum()} / {len(df)}\n")

Salaries disclosed: 62 / 585



In [75]:
# ---------- Date ----------
df["posted_at"] = pd.to_datetime(
    df["posted_at"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)


In [76]:
# ----------  Skills back to list ----------
df["skills_and_tools"] = df["skills_and_tools"].apply(
    lambda x: x if isinstance(x, list) else (
        [s.strip() for s in str(x).split(",") if s.strip()] if pd.notna(x) and x not in ["nan", ""] else []
    )
)

In [77]:
# Work type as list ----------
df["work_type"] = df["work_type"].apply(
    lambda x: [s.strip() for s in str(x).split(",") if s.strip()] if pd.notna(x) and x not in ["Unknown", "nan"] else []
)


In [78]:
# ---------- Categories ----------
translations = {
    "دوام كامل": "Full Time",
    "عمل من مقر الشركة": "On-Site",
    "ذو خبرة (غير إداري)": "Experienced (Non-Manager)",
    "مستوى مبتدئ (مبتدئ / خريج جديد)": "Entry Level (Junior Level / Fresh Grad)",
    "غير محدد": "Not Specified",
    "درجة البكالوريوس": "Bachelor's Degree",
}

# work_type is now a list, handle other cat cols as before
cat_cols = ["work_setting", "career_level", "education_level"]

for col in cat_cols:
    df[col] = df[col].replace(translations)
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].str.replace(
        r"'([A-Z])", lambda m: "'" + m.group(1).lower(), regex=True
    )

mode_setting = df["work_setting"][df["work_setting"] != "Unknown"].mode()[0]
df["work_setting"] = df["work_setting"].replace("Unknown", mode_setting)


In [79]:
# ---------- Location ----------
def split_location(x):
    parts = [p.strip() for p in str(x).split(",")]
    country = parts[-1] if len(parts) >= 1 else np.nan
    city = parts[-2] if len(parts) >= 2 else np.nan
    return city, country


df[["city", "country"]] = df["location"].apply(split_location).apply(pd.Series)


In [80]:
# ---------- Final ----------
print("Final shape:", df.shape)

Final shape: (585, 24)


In [81]:
# ---------- Save ----------
df.to_csv("clean_data.csv", index=False, encoding="utf-8-sig")

df.to_json(
    "clean_data.json",
    orient="records",
    indent=4,
    force_ascii=False,
    date_format="iso"
)

print("\nDone — files saved to outputs/")



Done — files saved to outputs/
